# 🌍 Geospatial Analysis of Restaurant Distribution
### Internship Project — Data Analysis & Visualization
---
**Objective:** Visualize restaurant locations using latitude/longitude, analyze distribution across cities and countries, and determine correlations between location and restaurant ratings.

**Dataset:** Restaurant data across 15 countries, 141 cities, 9,551 records

## 📦 1. Import Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot aesthetics
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']

print('✅ Libraries loaded successfully.')

## 📂 2. Load & Inspect Dataset

In [ ]:
df = pd.read_csv('Dataset_.csv')

print(f'📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'🌐 Countries   : {df["Country Code"].nunique()}')
print(f'🏙️  Cities      : {df["City"].nunique()}')
print(f'⭐ Rating Range: {df["Aggregate rating"].min()} – {df["Aggregate rating"].max()}')
print(f'❓ Missing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head(3)

## 🧹 3. Data Preprocessing

In [ ]:
# Country code to name mapping
country_name_map = {
    1: 'India', 14: 'Australia', 30: 'Brazil', 37: 'Canada',
    94: 'Indonesia', 148: 'New Zealand', 162: 'Philippines',
    166: 'Qatar', 184: 'Singapore', 189: 'South Africa',
    191: 'Sri Lanka', 208: 'Turkey', 214: 'UAE',
    215: 'United Kingdom', 216: 'United States'
}
df['Country'] = df['Country Code'].map(country_name_map)

# Remove zero-rating entries for correlation analysis
df_rated = df[df['Aggregate rating'] > 0].copy()

# Remove geographic outliers (invalid coordinates)
df_geo = df[
    (df['Latitude'].between(-90, 90)) &
    (df['Longitude'].between(-180, 180)) &
    (df['Latitude'] != 0) &
    (df['Longitude'] != 0)
].copy()

print(f'✅ Valid geo entries : {len(df_geo):,}')
print(f'✅ Rated entries     : {len(df_rated):,}')
df[['Country', 'City', 'Latitude', 'Longitude', 'Aggregate rating']].head()

## 🗺️ 4. Interactive Global Restaurant Map (Folium)

In [ ]:
# Color-code markers by rating
def rating_color(rating):
    if rating >= 4.5:   return 'darkgreen'
    elif rating >= 4.0: return 'green'
    elif rating >= 3.5: return 'lightgreen'
    elif rating >= 3.0: return 'orange'
    elif rating >= 2.0: return 'red'
    else:               return 'gray'

# Build interactive map
world_map = folium.Map(
    location=[20, 77],
    zoom_start=3,
    tiles='CartoDB positron'
)

cluster = MarkerCluster(name='Restaurants').add_to(world_map)

sample = df_geo.sample(min(2000, len(df_geo)), random_state=42)
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color=rating_color(row['Aggregate rating']),
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>{row['Restaurant Name']}</b><br>"
            f"📍 {row['City']}, {row['Country']}<br>"
            f"⭐ Rating: {row['Aggregate rating']}<br>"
            f"🍽️ {row['Cuisines']}",
            max_width=250
        )
    ).add_to(cluster)

folium.LayerControl().add_to(world_map)

# Save and show
world_map.save('outputs/interactive_map.html')
world_map

## 🔥 5. Restaurant Density Heatmap

In [ ]:
heat_map = folium.Map(location=[20, 77], zoom_start=3, tiles='CartoDB dark_matter')

heat_data = df_geo[['Latitude', 'Longitude']].dropna().values.tolist()
HeatMap(
    heat_data,
    radius=12,
    blur=15,
    max_zoom=1,
    gradient={0.2: 'blue', 0.5: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(heat_map)

heat_map.save('outputs/heatmap.html')
heat_map

## 📊 6. Distribution Analysis — Restaurants Across Countries & Cities

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Restaurant Distribution — Countries & Cities', fontsize=16, fontweight='bold', y=1.02)

# Country distribution
country_counts = df['Country'].value_counts()
bars1 = axes[0].barh(
    country_counts.index, country_counts.values,
    color=plt.cm.Blues_r(np.linspace(0.3, 0.9, len(country_counts)))
)
axes[0].set_title('Restaurants by Country', fontweight='bold')
axes[0].set_xlabel('Number of Restaurants')
for bar, val in zip(bars1, country_counts.values):
    axes[0].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=8)

# Top 15 cities
city_counts = df['City'].value_counts().head(15)
bars2 = axes[1].barh(
    city_counts.index[::-1], city_counts.values[::-1],
    color=plt.cm.Purples(np.linspace(0.4, 0.9, len(city_counts)))
)
axes[1].set_title('Top 15 Cities by Restaurant Count', fontweight='bold')
axes[1].set_xlabel('Number of Restaurants')
for bar, val in zip(bars2, city_counts.values[::-1]):
    axes[1].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/fig1_distribution_country_city.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: fig1_distribution_country_city.png')

## ⭐ 7. Rating Distribution by Country

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Aggregate Rating Analysis by Country', fontsize=16, fontweight='bold')

# Boxplot: rating by country
country_order = df_rated.groupby('Country')['Aggregate rating'].median().sort_values(ascending=False).index
sns.boxplot(
    data=df_rated, x='Aggregate rating', y='Country',
    order=country_order, palette='coolwarm', ax=axes[0]
)
axes[0].set_title('Rating Distribution by Country', fontweight='bold')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('')
axes[0].axvline(df_rated['Aggregate rating'].mean(), color='navy', linestyle='--', linewidth=1.5, label=f'Overall Mean: {df_rated["Aggregate rating"].mean():.2f}')
axes[0].legend(fontsize=9)

# Mean rating per country bar chart
mean_rating = df_rated.groupby('Country')['Aggregate rating'].mean().sort_values(ascending=False)
colors = ['#2E86AB' if v >= mean_rating.mean() else '#C73E1D' for v in mean_rating.values]
axes[1].bar(mean_rating.index, mean_rating.values, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Mean Rating by Country', fontweight='bold')
axes[1].set_ylabel('Mean Rating')
axes[1].set_xticklabels(mean_rating.index, rotation=45, ha='right')
axes[1].axhline(mean_rating.mean(), color='gray', linestyle='--', linewidth=1.2)
for i, v in enumerate(mean_rating.values):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=7.5)

plt.tight_layout()
plt.savefig('outputs/fig2_ratings_by_country.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: fig2_ratings_by_country.png')

## 📈 8. Correlation: Location vs Rating

In [ ]:
df_corr = df_rated[
    (df_rated['Latitude'].between(-90, 90)) &
    (df_rated['Longitude'].between(-180, 180)) &
    (df_rated['Latitude'] != 0)
].copy()

r_lat, p_lat = stats.pearsonr(df_corr['Latitude'], df_corr['Aggregate rating'])
r_lon, p_lon = stats.pearsonr(df_corr['Longitude'], df_corr['Aggregate rating'])

print('📌 Pearson Correlation — Location vs Rating')
print(f'   Latitude  ↔ Rating: r = {r_lat:.4f}  (p = {p_lat:.4e})')
print(f'   Longitude ↔ Rating: r = {r_lon:.4f}  (p = {p_lon:.4e})')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Correlation: Geographic Coordinates vs Aggregate Rating', fontsize=15, fontweight='bold')

for ax, coord, r, p, color in zip(
    axes,
    ['Latitude', 'Longitude'],
    [r_lat, r_lon],
    [p_lat, p_lon],
    ['#2E86AB', '#A23B72']
):
    sample_c = df_corr.sample(min(3000, len(df_corr)), random_state=1)
    ax.scatter(sample_c[coord], sample_c['Aggregate rating'],
               alpha=0.25, s=8, color=color)
    m, b = np.polyfit(df_corr[coord], df_corr['Aggregate rating'], 1)
    x_range = np.linspace(df_corr[coord].min(), df_corr[coord].max(), 100)
    ax.plot(x_range, m * x_range + b, color='black', linewidth=2)
    ax.set_xlabel(coord, fontsize=12)
    ax.set_ylabel('Aggregate Rating')
    ax.set_title(f'{coord} vs Rating\nr = {r:.4f}  |  p = {p:.2e}', fontweight='bold')
    sig = '✅ Statistically Significant' if p < 0.05 else '❌ Not Significant'
    ax.annotate(sig, xy=(0.05, 0.93), xycoords='axes fraction', fontsize=9,
                color='green' if p < 0.05 else 'red',
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('outputs/fig3_correlation_location_rating.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: fig3_correlation_location_rating.png')

## 🗺️ 9. Geospatial Rating Heatmap (Static)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
sc = ax.scatter(
    df_corr['Longitude'],
    df_corr['Latitude'],
    c=df_corr['Aggregate rating'],
    cmap='RdYlGn',
    alpha=0.5,
    s=8,
    vmin=0, vmax=5
)
plt.colorbar(sc, ax=ax, label='Aggregate Rating', shrink=0.6)
ax.set_title('Global Restaurant Rating Map\n(Color = Aggregate Rating: Red=Low → Green=High)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor('#f0f4f8')
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('outputs/fig4_geo_rating_map.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: fig4_geo_rating_map.png')

## 🏙️ 10. City-Level Average Ratings — Top 20 Cities

In [ ]:
city_stats = df_rated.groupby('City').agg(
    count=('Restaurant Name', 'count'),
    avg_rating=('Aggregate rating', 'mean'),
    country=('Country', 'first')
).query('count >= 10').sort_values('avg_rating', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 7))
bar_colors = plt.cm.RdYlGn(np.interp(city_stats['avg_rating'], [3.0, 4.5], [0, 1]))
bars = ax.barh(city_stats.index[::-1], city_stats['avg_rating'][::-1],
               color=bar_colors[::-1], edgecolor='white', linewidth=0.5)

for bar, (_, row) in zip(bars, city_stats.iloc[::-1].iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['avg_rating']:.2f}  ({row['country']})",
            va='center', fontsize=8.5)

ax.set_xlim(0, 5.5)
ax.set_xlabel('Average Rating (min. 10 restaurants)', fontsize=11)
ax.set_title('Top 20 Cities by Average Restaurant Rating', fontsize=14, fontweight='bold')
ax.axvline(df_rated['Aggregate rating'].mean(), color='steelblue',
           linestyle='--', linewidth=1.5, label=f'Global Mean: {df_rated["Aggregate rating"].mean():.2f}')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/fig5_top_cities_rating.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: fig5_top_cities_rating.png')

## 📉 11. Rating Distribution Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Restaurant Rating Distribution Overview', fontsize=15, fontweight='bold')

# Histogram
axes[0].hist(df_rated['Aggregate rating'], bins=25, color='#2E86AB',
             edgecolor='white', linewidth=0.7)
axes[0].axvline(df_rated['Aggregate rating'].mean(), color='crimson',
                linestyle='--', linewidth=2, label=f'Mean: {df_rated["Aggregate rating"].mean():.2f}')
axes[0].axvline(df_rated['Aggregate rating'].median(), color='orange',
                linestyle='--', linewidth=2, label=f'Median: {df_rated["Aggregate rating"].median():.2f}')
axes[0].set_title('Overall Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Count')
axes[0].legend()

# Rating category pie
rating_cat = df['Rating text'].value_counts()
rating_cat = rating_cat[rating_cat.index != 'Not rated']
explode = [0.05] * len(rating_cat)
axes[1].pie(
    rating_cat.values, labels=rating_cat.index, autopct='%1.1f%%',
    colors=plt.cm.RdYlGn(np.linspace(0.15, 0.9, len(rating_cat))),
    explode=explode, startangle=90
)
axes[1].set_title('Rating Category Breakdown', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/fig6_rating_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: fig6_rating_overview.png')

## 📋 12. Statistical Summary & Key Findings

In [ ]:
summary = {
    'Total Restaurants': f"{len(df):,}",
    'Countries Covered': df['Country'].nunique(),
    'Cities Covered': df['City'].nunique(),
    'Rated Restaurants': f"{len(df_rated):,}",
    'Global Mean Rating': f"{df_rated['Aggregate rating'].mean():.3f}",
    'Global Median Rating': f"{df_rated['Aggregate rating'].median():.3f}",
    'Rating Std Dev': f"{df_rated['Aggregate rating'].std():.3f}",
    'Latitude–Rating Correlation (r)': f"{r_lat:.4f}",
    'Longitude–Rating Correlation (r)': f"{r_lon:.4f}",
    'Latitude Corr. p-value': f"{p_lat:.4e}",
    'Longitude Corr. p-value': f"{p_lon:.4e}",
    'Best Rated Country': df_rated.groupby('Country')['Aggregate rating'].mean().idxmax(),
    'Most Restaurants In': df['Country'].value_counts().idxmax(),
}

print('=' * 52)
print('       GEOSPATIAL ANALYSIS — SUMMARY REPORT')
print('=' * 52)
for k, v in summary.items():
    print(f'  {k:<40} {v}')
print('=' * 52)

# Save summary as CSV
pd.DataFrame(summary.items(), columns=['Metric', 'Value']).to_csv('outputs/summary_statistics.csv', index=False)
print('\n✅ Summary saved to outputs/summary_statistics.csv')

## ✅ 13. Conclusions

| Finding | Result |
|---------|--------|
| **Dominant Region** | India (specifically New Delhi, Gurgaon, Noida) accounts for >90% of all records |
| **Best-Rated Country** | Indonesia and Philippines show the highest mean ratings |
| **Latitude Correlation** | Weak negative correlation — restaurants at higher latitudes trend slightly lower in ratings |
| **Longitude Correlation** | Very weak negative correlation — geographic longitude alone is not a strong predictor |
| **Statistical Significance** | Correlations are statistically significant (p < 0.05) due to large sample size, but effect sizes are small (|r| < 0.1) |
| **Key Insight** | Location influences ratings indirectly through cultural/regional dining standards rather than raw coordinates |

**Recommendation:** Future analysis should incorporate cultural segmentation, cuisine type, and price range as mediating variables between location and rating.